# Full-2024 training and 2025 evaluation

Fit the basic, neighbour-mean, and neighbour-shock pipelines on all of 2024, then evaluate rolling one-hour-ahead predictions during 2025.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer
import geopandas as gpd
import data_tools as data


In [ ]:
df_basic = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features.parquet"))
df_meanPU = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features_neighbor_mean.parquet"))
df_shock = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features_neighbor_shock.parquet"))
df_test_basic = pd.read_parquet(str(project_root / "data/yellow_taxi_2025_features.parquet"))
df_test_meanPU = pd.read_parquet(str(project_root / "data/yellow_taxi_2025_features_neighbor_mean.parquet"))
df_test_shock = pd.read_parquet(str(project_root / "data/yellow_taxi_2025_features_neighbor_shock.parquet"))


In [ ]:
X_basic = df_basic.copy()
X_meanPU = df_meanPU.copy()
X_shock = df_shock.copy()


In [ ]:
# Base features: train on all of 2024 and evaluate on all of 2025.
X_train_basic = X_basic.loc[X_basic["tpep_pickup_datetime"].dt.year == 2024].copy()
X_test_basic = df_test_basic.loc[df_test_basic["tpep_pickup_datetime"].dt.year == 2025].copy()
y_train_basic = X_train_basic["trip_count"]
y_test_basic = X_test_basic["trip_count"]
train_basic = X_train_basic.drop(columns = ["trip_count", "tpep_pickup_datetime"])
test_basic = X_test_basic.drop(columns = ["trip_count", "tpep_pickup_datetime"])

assert len(X_train_basic) == 366 * 24 * 263
assert len(X_test_basic) == 365 * 24 * 263
assert X_train_basic["tpep_pickup_datetime"].max() < X_test_basic["tpep_pickup_datetime"].min()
assert train_basic.index.equals(y_train_basic.index)
assert test_basic.index.equals(y_test_basic.index)
print("Base features: training on 12 months of 2024; testing on 12 months of 2025.")


In [ ]:
# Neighbour mean: train on all of 2024 and evaluate on all of 2025.
X_train_meanPU = X_meanPU.loc[X_meanPU["tpep_pickup_datetime"].dt.year == 2024].copy()
X_test_meanPU = df_test_meanPU.loc[df_test_meanPU["tpep_pickup_datetime"].dt.year == 2025].copy()
y_train_meanPU = X_train_meanPU["trip_count"]
y_test_meanPU = X_test_meanPU["trip_count"]
train_meanPU = X_train_meanPU.drop(columns = ["trip_count", "tpep_pickup_datetime"])
test_meanPU = X_test_meanPU.drop(columns = ["trip_count", "tpep_pickup_datetime"])

assert len(X_train_meanPU) == 366 * 24 * 263
assert len(X_test_meanPU) == 365 * 24 * 263
assert X_train_meanPU["tpep_pickup_datetime"].max() < X_test_meanPU["tpep_pickup_datetime"].min()
assert train_meanPU.index.equals(y_train_meanPU.index)
assert test_meanPU.index.equals(y_test_meanPU.index)
print("Neighbour mean: training on 12 months of 2024; testing on 12 months of 2025.")


In [ ]:
# Neighbour shock: train on all of 2024 and evaluate on all of 2025.
X_train_shock = X_shock.loc[X_shock["tpep_pickup_datetime"].dt.year == 2024].copy()
X_test_shock = df_test_shock.loc[df_test_shock["tpep_pickup_datetime"].dt.year == 2025].copy()
y_train_shock = X_train_shock["trip_count"]
y_test_shock = X_test_shock["trip_count"]
train_shock = X_train_shock.drop(columns = ["trip_count", "tpep_pickup_datetime"])
test_shock = X_test_shock.drop(columns = ["trip_count", "tpep_pickup_datetime"])

assert len(X_train_shock) == 366 * 24 * 263
assert len(X_test_shock) == 365 * 24 * 263
assert X_train_shock["tpep_pickup_datetime"].max() < X_test_shock["tpep_pickup_datetime"].min()
assert train_shock.index.equals(y_train_shock.index)
assert test_shock.index.equals(y_test_shock.index)
print("Neighbour shock: training on 12 months of 2024; testing on 12 months of 2025.")


In [ ]:
model_basic = data.model(train_basic)
model_spatial = data.model(train_meanPU)
model_shock = data.model(train_shock)

model_basic.fit(train_basic, y_train_basic)
model_spatial.fit(train_meanPU, y_train_meanPU)
model_shock.fit(train_shock, y_train_shock)


In [ ]:
# Inputs for 2025 use the same feature columns and order as full-year training.
X_2025_basic = test_basic[train_basic.columns]
X_2025_mean = test_meanPU[train_meanPU.columns]
X_2025_shock = test_shock[train_shock.columns]

y_pred_basic = model_basic.predict(X_2025_basic)
y_pred_mean = model_spatial.predict(X_2025_mean)
y_pred_shock = model_shock.predict(X_2025_shock)


In [ ]:
print("Basic — full-year 2025 test performance (trained on all of 2024)")
print(f"MAE:  {mean_absolute_error(y_test_basic, y_pred_basic):.4f}")
print(f"RMSE: {root_mean_squared_error(y_test_basic, y_pred_basic):.4f}")
print("\nPredicted pickups per zone per hour")
print(f"Mean:            {y_pred_basic.mean():.4f}")
print(f"Median:          {np.median(y_pred_basic):.4f}")
print(f"95th percentile: {np.percentile(y_pred_basic, 95):.4f}")
print(f"99th percentile: {np.percentile(y_pred_basic, 99):.4f}")
print(f"Maximum:         {np.max(y_pred_basic):.4f}")


In [ ]:
print("Neighbour mean — full-year 2025 test performance (trained on all of 2024)")
print(f"MAE:  {mean_absolute_error(y_test_meanPU, y_pred_mean):.4f}")
print(f"RMSE: {root_mean_squared_error(y_test_meanPU, y_pred_mean):.4f}")
print("\nPredicted pickups per zone per hour")
print(f"Mean:            {y_pred_mean.mean():.4f}")
print(f"Median:          {np.median(y_pred_mean):.4f}")
print(f"95th percentile: {np.percentile(y_pred_mean, 95):.4f}")
print(f"99th percentile: {np.percentile(y_pred_mean, 99):.4f}")
print(f"Maximum:         {np.max(y_pred_mean):.4f}")


In [ ]:
print("Neighbour shock — full-year 2025 test performance (trained on all of 2024)")
print(f"MAE:  {mean_absolute_error(y_test_shock, y_pred_shock):.4f}")
print(f"RMSE: {root_mean_squared_error(y_test_shock, y_pred_shock):.4f}")
print("\nPredicted pickups per zone per hour")
print(f"Mean:            {y_pred_shock.mean():.4f}")
print(f"Median:          {np.median(y_pred_shock):.4f}")
print(f"95th percentile: {np.percentile(y_pred_shock, 95):.4f}")
print(f"99th percentile: {np.percentile(y_pred_shock, 99):.4f}")
print(f"Maximum:         {np.max(y_pred_shock):.4f}")


## Monthly 2025 performance


In [ ]:
def train_basic_monthly(m):
    train_month = df_test_basic[df_test_basic.month == m]
    return model_basic.predict(train_month)

def train_mean_monthly(m):
    train_month = df_test_meanPU[df_test_meanPU.month == m]
    return model_spatial.predict(train_month)

def train_shock_monthly(m):
    train_month = df_test_shock[df_test_shock.month == m]
    return model_shock.predict(train_month)

comp_table = [[]for _ in range(12)]
comp_rsme = [[]for _ in range(12)]
for m in range(1, 13):
    y_month = df_test_basic.trip_count[df_test_basic.month == m]
    y_pred_basic_m = train_basic_monthly(m)
    y_pred_mean_m = train_mean_monthly(m)
    y_pred_shock_m = train_shock_monthly(m)
    mae_basic = mean_absolute_error(y_pred_basic_m, y_month)
    mae_mean = mean_absolute_error(y_pred_mean_m, y_month)
    mae_shock = mean_absolute_error(y_pred_shock_m, y_month)
    rsme_basic = root_mean_squared_error(y_pred_basic_m, y_month)
    rsme_mean = root_mean_squared_error(y_pred_mean_m, y_month)
    rsme_shock = root_mean_squared_error(y_pred_shock_m, y_month)
    comp_table[m-1] = [mae_basic, mae_mean, mae_shock]
    comp_rsme[m-1] = [rsme_basic, rsme_mean, rsme_shock]


In [ ]:
months_2025 = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
monthly_mae = pd.DataFrame(
    comp_table,
    index = pd.Index(months_2025, name = "Month (2025)"),
    columns = ["Base", "Neighbour mean", "Neighbour shock"],
)

# The best value for each month is highlighted in green.
style_mae = (
    monthly_mae.style
    .format("{:.4f}")
    .highlight_min(axis = 1, props = "background-color: #dcfce7; color: #14532d; font-weight: bold;")
    .set_caption("Monthly MAE — 2025 test — lower is better")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("text-align", "left"), ("padding", "10px 0")]},
        {"selector": "th", "props": [("padding", "8px 12px"), ("text-align", "left")]},
        {"selector": "td", "props": [("padding", "8px 12px"), ("text-align", "right")]},
    ])
)
style_mae


In [ ]:
months_2025 = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
monthly_rmse = pd.DataFrame(
    comp_rsme,
    index = pd.Index(months_2025, name = "Month (2025)"),
    columns = ["Base", "Neighbour mean", "Neighbour shock"],
)

# The best value for each month is highlighted in green.
style_rmse = (
    monthly_rmse.style
    .format("{:.4f}")
    .highlight_min(axis = 1, props = "background-color: #dcfce7; color: #14532d; font-weight: bold;")
    .set_caption("Monthly RMSE — 2025 test — lower is better")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("text-align", "left"), ("padding", "10px 0")]},
        {"selector": "th", "props": [("padding", "8px 12px"), ("text-align", "left")]},
        {"selector": "td", "props": [("padding", "8px 12px"), ("text-align", "right")]},
    ])
)
style_rmse


In [ ]:
from pathlib import Path

# Preserve the 2025 dataset row order; export only the three predictions.
predictions_2025 = pd.DataFrame({
    "pred_basic": np.asarray(y_pred_basic),
    "pred_mean": np.asarray(y_pred_mean),
    "pred_shock": np.asarray(y_pred_shock),
})
assert len(predictions_2025) == len(df_test_basic)
assert predictions_2025.notna().all().all()
prediction_project_dir = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
prediction_output_path = prediction_project_dir / "data" / "predictions_2025.parquet"
predictions_2025.to_parquet(prediction_output_path, index = False)
print(f"Saved {len(predictions_2025):,} rows and three prediction columns to {prediction_output_path}")
